# 🌍 Avant / Après : Sites d'extraction et de remblais

Ce notebook utilise **Google Earth Engine** via **geemap** pour visualiser l'évolution de sites d'extraction (carrières, cimenteries) et de remblais (terres gagnées sur l'eau) entre une année de référence et l'année la plus récente disponible dans GEE.

Les images proviennent des satellites **Landsat** (résolution 30 m). Le contrôle **split-panel** permet de faire glisser un curseur pour comparer les deux époques côte à côte.

---

## Sites couverts

### 🪨 Sites d'extraction
| # | Site | Localisation | Type | Avant | Après |
|---|------|-------------|------|-------|-------|
| 1 | Carrare | Italie | Carrière de marbre | 1984 | récent |
| 2 | Salève | France/Suisse | Carrière & sablière | 1984 | récent |
| 3 | La Araña – Cementos Cosmos | Espagne | Carrière & cimenterie | 1984 | récent |
| 4 | Mont-Saint-Hilaire | Québec, Canada | Carrière de granulats | 1984 | récent |
| 5 | Siam Cement | Lampang, Thaïlande | Cimenterie | 1984 | récent |

### 🏝️ Sites de remblais
| # | Site | Localisation | Avant | Après |
|---|------|-------------|-------|-------|
| 6 | Maldives – Malé | Océan Indien | 1984 | récent |
| 7 | Genève – Baby-Plage | Suisse | 1984 | récent |
| 8 | Singapour | Asie du SE | 1984 | récent |
| 9 | Lagos – Eko Atlantic City | Nigeria | 2005 | récent |
| 10 | Cát Bà – Cái Giá | Vietnam | 2000 | récent |

## 1. Installation et imports

In [1]:
# Décommenter si nécessaire :
# !pip install geemap earthengine-api -q

import ee
import geemap
import os
import math
from pathlib import Path

print(f"geemap version : {geemap.__version__}")

geemap version : 0.37.2


## 2. Authentification Google Earth Engine

In [2]:
# Première utilisation : ee.Authenticate() ouvre une page d'autorisation dans le navigateur
# ee.Authenticate()
ee.Initialize(project='geemap-496017')  # ← Remplacez par votre projet GEE

## 3. Fonctions utilitaires

### 3.1 Sélection du satellite et de la ROI

- **1984–1998** → Landsat 5 TM (collection 2, niveau 2)
- **1999–2012** → Landsat 7 ETM+ (collection 2, niveau 2)
- **2013–présent** → Landsat 8/9 OLI (collection 2, niveau 2)

La ROI est un **rectangle** défini par une demi-largeur et une demi-hauteur en mètres, centré sur le site. Cela permet d'adapter précisément la zone affichée à l'échelle de la carte sans introduire un artefact circulaire.

In [3]:
# ─── Sélection du satellite ───────────────────────────────────────────────────

def get_landsat_info(year):
    """Retourne (collection_id, bandes_RGB, nom_satellite) pour une année donnée."""
    if year <= 1998:
        return 'LANDSAT/LT05/C02/T1_L2', ['SR_B3', 'SR_B2', 'SR_B1'], 'Landsat 5'
    elif year <= 2012:
        return 'LANDSAT/LE07/C02/T1_L2', ['SR_B3', 'SR_B2', 'SR_B1'], 'Landsat 7'
    else:
        return 'LANDSAT/LC08/C02/T1_L2', ['SR_B4', 'SR_B3', 'SR_B2'], 'Landsat 8/9'


def apply_scale_factors(image):
    """Facteurs d'échelle Landsat Collection 2 Level-2 Surface Reflectance."""
    return image.addBands(
        image.select('SR_B.*').multiply(0.0000275).add(-0.2),
        overwrite=True
    )


# ─── ROI rectangulaire ────────────────────────────────────────────────────────

def make_rect_roi(lon, lat, half_w_m, half_h_m):
    """
    Crée un rectangle ee.Geometry centré sur (lon, lat).

    Paramètres
    ----------
    lon, lat   : centre en degrés décimaux
    half_w_m   : demi-largeur en mètres (axe Est-Ouest)
    half_h_m   : demi-hauteur en mètres (axe Nord-Sud)
    """
    # Conversion mètres → degrés (approximation locale)
    lat_rad   = math.radians(lat)
    delta_lon = (half_w_m / (111_320 * math.cos(lat_rad)))
    delta_lat = (half_h_m / 110_540)
    return ee.Geometry.Rectangle([
        lon - delta_lon, lat - delta_lat,
        lon + delta_lon, lat + delta_lat
    ])


# ─── Mosaïque Landsat ─────────────────────────────────────────────────────────

# Plage temporelle couverte par Landsat dans GEE
LANDSAT_FIRST_YEAR = 1984   # début de Landsat 5 Collection 2
LANDSAT_LAST_YEAR  = 2025   # dernière année complète disponible dans GEE


def _find_year_with_images(collection_id, roi, candidate_years, max_cloud=20):
    """
    Parcourt candidate_years dans l'ordre et retourne le premier qui possède
    au moins une image avec CLOUD_COVER < max_cloud sur la fenêtre avril–octobre.
    Tente ensuite toute l'année avec CC < 40 % comme fallback.
    """
    for yr in candidate_years:
        col = (
            ee.ImageCollection(collection_id)
            .filterBounds(roi)
            .filterDate(f'{yr}-04-01', f'{yr}-10-31')
            .filter(ee.Filter.lt('CLOUD_COVER', max_cloud))
        )
        if col.size().getInfo() > 0:
            return yr
    # Fallback : essayer avec seuil élargi
    for yr in candidate_years:
        col = (
            ee.ImageCollection(collection_id)
            .filterBounds(roi)
            .filterDate(f'{yr}-01-01', f'{yr}-12-31')
            .filter(ee.Filter.lt('CLOUD_COVER', 40))
        )
        if col.size().getInfo() > 0:
            print(f"  ⚠️  Année {yr} : filtre élargi (CC < 40 %, année complète)")
            return yr
    return candidate_years[0]  # dernier recours


def get_latest_year(collection_id, roi, max_cloud=20):
    """
    Retourne l'année la plus récente disponible dans la collection
    (recherche en remontant depuis LANDSAT_LAST_YEAR).
    """
    candidates = list(range(LANDSAT_LAST_YEAR, LANDSAT_LAST_YEAR - 6, -1))
    return _find_year_with_images(collection_id, roi, candidates, max_cloud)


def get_earliest_year(collection_id, roi, hint_year, max_cloud=20):
    """
    Retourne l'année la plus ancienne disponible dans la collection
    à partir de hint_year (recherche en avançant depuis hint_year).
    Utile pour trouver la première image exploitable au lieu d'utiliser
    une année codée en dur.
    """
    # On cherche sur une fenêtre de 5 ans à partir de l'année indiquée
    candidates = list(range(hint_year, hint_year + 6))
    return _find_year_with_images(collection_id, roi, candidates, max_cloud)


def get_landsat_mosaic(lon, lat, year, half_w_m, half_h_m,
                       find_latest=False, find_earliest=False):
    """
    Crée une mosaïque médiane Landsat pour un site, une emprise et une année.

    Modes
    -----
    find_latest=True  : ignore 'year', utilise Landsat 8/9, cherche l'année
                        la plus récente disponible dans GEE.
    find_earliest=True: ignore 'year', utilise le satellite adapté à hint_year,
                        cherche la première année disponible à partir de 'year'.
    (défaut)          : utilise exactement 'year' avec le satellite correspondant.

    Retourne (image, bandes_RGB, satellite, année_effective, roi)
    """
    roi = make_rect_roi(lon, lat, half_w_m, half_h_m)

    if find_latest:
        collection_id = 'LANDSAT/LC08/C02/T1_L2'
        bands         = ['SR_B4', 'SR_B3', 'SR_B2']
        satellite     = 'Landsat 8/9'
        year          = get_latest_year(collection_id, roi)

    elif find_earliest:
        # Sélectionner le bon satellite pour l'époque cible
        collection_id, bands, satellite = get_landsat_info(year)
        year = get_earliest_year(collection_id, roi, hint_year=year)
        # Mettre à jour satellite/bands si l'année effective a changé
        collection_id, bands, satellite = get_landsat_info(year)

    else:
        collection_id, bands, satellite = get_landsat_info(year)

    start = f'{year}-04-01'
    end   = f'{year}-10-31'

    col = (
        ee.ImageCollection(collection_id)
        .filterBounds(roi)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUD_COVER', 20))
        .map(apply_scale_factors)
        .select(bands)
    )

    # Fallback : élargir à toute l'année si pas assez d'images
    if col.size().getInfo() == 0:
        col = (
            ee.ImageCollection(collection_id)
            .filterBounds(roi)
            .filterDate(f'{year}-01-01', f'{year}-12-31')
            .filter(ee.Filter.lt('CLOUD_COVER', 40))
            .map(apply_scale_factors)
            .select(bands)
        )
        print(f"  ⚠️  Peu d'images : filtre élargi à l'année complète (CC < 40 %)")

    mosaic = col.median().clip(roi)
    return mosaic, bands, satellite, year, roi


# Paramètres de visualisation communs (couleurs naturelles)
VIS = {'min': 0.0, 'max': 0.3, 'gamma': 1.4}

### 3.2 Carte split-panel

In [4]:
def make_split_map(site_name, lon, lat, year_before,
                   half_w_m=4000, half_h_m=3000, zoom=13):
    """
    Génère une carte geemap avec curseur split avant/après.

    Paramètres
    ----------
    site_name   : nom du site (affiché dans les labels)
    lon, lat    : centre du site en degrés décimaux
    year_before : année de référence (côté gauche)
    half_w_m    : demi-largeur de l'emprise rectangulaire en mètres
    half_h_m    : demi-hauteur de l'emprise rectangulaire en mètres
    zoom        : niveau de zoom initial de la carte

    Retourne
    --------
    m           : objet geemap.Map prêt à afficher
    meta        : dict avec les métadonnées (années, satellites, roi)
    """
    print(f"\n🔍  {site_name}")

    print(f"    Avant (à partir de {year_before})…", end=' ', flush=True)
    img_b, bands_b, sat_b, yr_b, roi = get_landsat_mosaic(
        lon, lat, year_before, half_w_m, half_h_m, find_earliest=True
    )
    print(f"OK → {yr_b} ({sat_b})")

    print(f"    Après (année la plus récente)…", end=' ', flush=True)
    img_a, bands_a, sat_a, yr_a, _   = get_landsat_mosaic(
        lon, lat, 9999, half_w_m, half_h_m, find_latest=True
    )
    print(f"OK → {yr_a} ({sat_a})")

    left_layer  = geemap.ee_tile_layer(img_b, VIS, f"{yr_b} – {sat_b}")
    right_layer = geemap.ee_tile_layer(img_a, VIS, f"{yr_a} – {sat_a}")

    m = geemap.Map(center=[lat, lon], zoom=zoom, height='560px')
    m.split_map(
        left_layer,
        right_layer,
        left_label=f"◀ {yr_b}",
        right_label=f"{yr_a} ▶"
    )

    meta = dict(
        site_name=site_name, lon=lon, lat=lat,
        yr_b=yr_b, yr_a=yr_a,
        sat_b=sat_b, sat_a=sat_a,
        img_b=img_b, img_a=img_a,
        roi=roi, half_w_m=half_w_m, half_h_m=half_h_m
    )
    return m, meta

---
## 4. 🪨 Sites d'extraction

### 4.1 Carrare, Italie – Carrière de marbre

> Les Alpes Apuanes abritent les carrières de marbre blanc les plus célèbres au monde. En 40 ans, l'extraction intensive a élargi les fronts de taille, visibles comme des taches blanches caractéristiques sur Landsat.

In [5]:
m_carrare, meta_carrare = make_split_map(
    site_name   = "Carrare – Carrière de marbre (Italie)",
    lon         = 10.1368,   # 10°08'12.5"E
    lat         = 44.0981,   # 44°05'53.1"N
    year_before = 1984,
    half_w_m    = 5000,      # emprise : ~10 km × 8 km
    half_h_m    = 4000,
    zoom        = 13
)
m_carrare


🔍  Carrare – Carrière de marbre (Italie)
    Avant (à partir de 1984)… OK → 1984 (Landsat 5)
    Après (année la plus récente)… OK → 2025 (Landsat 8/9)


Map(center=[44.0981, 10.1368], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zo…

### 4.2 Salève, France/Suisse – Carrière & sablière

> Le mont Salève, aux portes de Genève, abrite plusieurs carrières sur ses flancs. La résolution Landsat 30 m permet de distinguer les zones dénudées.

In [6]:
m_saleve, meta_saleve = make_split_map(
    site_name   = "Salève – Carrière & sablière (France/Suisse)",
    lon         = 6.1898,    # 6°11'23.4"E
    lat         = 46.1591,   # 46°09'32.6"N
    year_before = 1984,
    half_w_m    = 3500,
    half_h_m    = 2500,
    zoom        = 13
)
m_saleve


🔍  Salève – Carrière & sablière (France/Suisse)
    Avant (à partir de 1984)… OK → 1984 (Landsat 5)
    Après (année la plus récente)… OK → 2025 (Landsat 8/9)


Map(center=[46.1591, 6.1898], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

### 4.3 La Araña – Cementos Cosmos, Espagne

> Cimenterie et carrière calcaire à l'est de Málaga. L'exploitation s'est étendue vers les falaises côtières sur les quatre dernières décennies.

In [7]:
m_arana, meta_arana = make_split_map(
    site_name   = "La Araña – Cementos Cosmos (Málaga, Espagne)",
    lon         = -4.3255,   # 4°19'31.6"W
    lat         = 36.7189,   # 36°43'08.0"N
    year_before = 1984,
    half_w_m    = 3500,
    half_h_m    = 2500,
    zoom        = 13
)
m_arana


🔍  La Araña – Cementos Cosmos (Málaga, Espagne)
    Avant (à partir de 1984)… OK → 1984 (Landsat 5)
    Après (année la plus récente)… OK → 2025 (Landsat 8/9)


Map(center=[36.7189, -4.3255], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zo…

### 4.4 Mont-Saint-Hilaire, Québec, Canada – Carrière de granulats

> L'exploitation du mont Saint-Hilaire (roche intrusive) a débuté avant 1984 ; l'évolution en surface reste modérée sur la période étudiée, l'extraction se poursuivant principalement en profondeur. À titre indicatif.

In [8]:
m_msh, meta_msh = make_split_map(
    site_name   = "Mont-Saint-Hilaire – Carrière de granulats (Québec)",
    lon         = -73.1833,
    lat         = 45.5500,
    year_before = 1984,
    half_w_m    = 3000,
    half_h_m    = 2000,
    zoom        = 13
)
m_msh


🔍  Mont-Saint-Hilaire – Carrière de granulats (Québec)
    Avant (à partir de 1984)… OK → 1984 (Landsat 5)
    Après (année la plus récente)… OK → 2025 (Landsat 8/9)


Map(center=[45.55, -73.1833], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

### 4.5 Siam Cement, province de Lampang, Thaïlande

> L'un des plus grands complexes cimentiers d'Asie du Sud-Est. La carrière calcaire adossée à l'usine a connu une expansion considérable depuis 1984.

In [9]:
m_siam, meta_siam = make_split_map(
    site_name   = "Siam Cement – Lampang (Thaïlande)",
    lon         = 99.5868,   # 99°35'12.3"E
    lat         = 18.5491,   # 18°32'56.7"N
    year_before = 1984,
    half_w_m    = 5000,
    half_h_m    = 3500,
    zoom        = 13
)
m_siam


🔍  Siam Cement – Lampang (Thaïlande)
    Avant (à partir de 1984)… OK → 1988 (Landsat 5)
    Après (année la plus récente)… OK → 2025 (Landsat 8/9)


Map(center=[18.5491, 99.5868], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zo…

---
## 5. 🏝️ Sites de remblais

### 5.1 Maldives – Atolls à l'ouest de Malé

> La création de Hulhumalé et d'autres îles artificielles aux Maldives pour faire face à l'élévation du niveau de la mer est spectaculaire : de nouvelles masses terrestres ont littéralement émergé de l'océan.

In [10]:
m_maldives, meta_maldives = make_split_map(
    site_name   = "Maldives – Malé, atolls à l'ouest",
    lon         = 73.4760,   # 73°28'33.4"E
    lat         = 4.1742,    # 4°10'27.2"N
    year_before = 1984,
    half_w_m    = 8000,
    half_h_m    = 5000,
    zoom        = 12
)
m_maldives


🔍  Maldives – Malé, atolls à l'ouest
    Avant (à partir de 1984)…   ⚠️  Peu d'images : filtre élargi à l'année complète (CC < 40 %)
OK → 1984 (Landsat 5)
    Après (année la plus récente)… OK → 2025 (Landsat 8/9)


Map(center=[4.1742, 73.476], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

### 5.2 Genève – Baby-Plage (Lac Léman)

> La plage artificielle de la Baby-Plage à Genève et les aménagements riverains ont modifié le trait de côte du lac. La résolution Landsat (30 m) reste à la limite de détection pour ce site.

In [11]:
m_geneve, meta_geneve = make_split_map(
    site_name   = "Genève – Baby-Plage (Lac Léman)",
    lon         = 6.1648,    # 6°09'52.8"E
    lat         = 46.2093,   # 46°12'33.3"N
    year_before = 1984,
    half_w_m    = 2000,
    half_h_m    = 1500,
    zoom        = 14
)
m_geneve


🔍  Genève – Baby-Plage (Lac Léman)
    Avant (à partir de 1984)… OK → 1984 (Landsat 5)
    Après (année la plus récente)… OK → 2025 (Landsat 8/9)


Map(center=[46.2093, 6.1648], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

### 5.3 Singapour – Polders côtiers (Tuas)

> Singapour a agrandi sa superficie de plus de 25 % grâce à des remblais massifs. Sur la côte ouest (Tuas), les transformations sont parmi les plus spectaculaires au monde.

In [12]:
m_singapour, meta_singapour = make_split_map(
    site_name   = "Singapour – Remblais côtiers (Tuas, côte ouest)",
    lon         = 103.7123,  # 103°42'44.6"E
    lat         = 1.2790,    # 1°16'44.4"N
    year_before = 1984,
    half_w_m    = 8000,
    half_h_m    = 5000,
    zoom        = 12
)
m_singapour


🔍  Singapour – Remblais côtiers (Tuas, côte ouest)
    Avant (à partir de 1984)… OK → 1989 (Landsat 5)
    Après (année la plus récente)… OK → 2025 (Landsat 8/9)


Map(center=[1.279, 103.7123], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

### 5.4 Lagos – Eko Atlantic City, Nigeria

> Eko Atlantic City est une ville nouvelle construite sur une presqu'île artificielle remblayée dans le golfe de Guinée, à partir de 2005. Les travaux sont encore en cours.

In [13]:
m_lagos, meta_lagos = make_split_map(
    site_name   = "Lagos – Eko Atlantic City (Nigeria)",
    lon         = 3.4347,    # 3°26'04.8"E
    lat         = 6.4139,    # 6°24'49.9"N
    year_before = 2005,
    half_w_m    = 5000,
    half_h_m    = 4000,
    zoom        = 13
)
m_lagos


🔍  Lagos – Eko Atlantic City (Nigeria)
    Avant (à partir de 2005)… OK → 2006 (Landsat 7)
    Après (année la plus récente)… OK → 2024 (Landsat 8/9)


Map(center=[6.4139, 3.4347], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

### 5.5 Khu đô thị Cái Giá – Cát Bà, Vietnam

> Dans la baie d'Ha Long, l'île de Cát Bà a vu émerger d'importants remblais pour des projets touristiques et immobiliers depuis les années 2000.

In [14]:
m_catba, meta_catba = make_split_map(
    site_name   = "Cát Bà – Khu đô thị Cái Giá (Vietnam)",
    lon         = 107.0305,  # 107°01'49.9"E
    lat         = 20.7356,   # 20°44'08.0"N
    year_before = 2000,
    half_w_m    = 5000,
    half_h_m    = 3500,
    zoom        = 13
)
m_catba


🔍  Cát Bà – Khu đô thị Cái Giá (Vietnam)
    Avant (à partir de 2000)… OK → 2000 (Landsat 7)
    Après (année la plus récente)… OK → 2025 (Landsat 8/9)


Map(center=[20.7356, 107.0305], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

---
## 6. 📸 Export haute résolution (pour impression)

Les images avant/après sont exportées en **GeoTIFF** (résolution native Landsat 30 m) puis converties en **PNG haute résolution** (300 dpi, format A4 paysage) pour l'impression.

Deux modes d'export sont proposés :
- **`export_geotiff()`** : export direct vers le disque local via `geemap.ee_export_image()`.
- **`export_to_drive()`** : export vers Google Drive pour les très grandes zones (tâche GEE asynchrone).

Le répertoire de sortie est `./exports/`.

In [ ]:
import subprocess

EXPORT_DIR = Path('./exports')
EXPORT_DIR.mkdir(exist_ok=True)


def safe_filename(name):
    """Transforme un nom de site en nom de fichier valide."""
    return (
        name.lower()
        .replace(' ', '_')
        .replace('–', '-')
        .replace('/', '-')
        .replace('(', '').replace(')', '')
        .replace(',', '')
        .replace("'", '')
        .replace('é', 'e').replace('è', 'e').replace('ê', 'e')
        .replace('â', 'a').replace('à', 'a')
        .replace('î', 'i').replace('ô', 'o').replace('û', 'u')
    )


def export_site_images(meta, scale=30, crs='EPSG:4326'):
    """
    Exporte les images avant/après d'un site en GeoTIFF + PNG haute résolution.

    Paramètres
    ----------
    meta   : dict retourné par make_split_map()
    scale  : résolution en mètres (30 = natif Landsat)
    crs    : système de projection de sortie
    """
    slug = safe_filename(meta['site_name'])
    roi  = meta['roi']

    for tag, img, yr, sat in [
        ('avant', meta['img_b'], meta['yr_b'], meta['sat_b']),
        ('apres', meta['img_a'], meta['yr_a'], meta['sat_a']),
    ]:
        tif_path = EXPORT_DIR / f"{slug}_{tag}_{yr}.tif"
        png_path = EXPORT_DIR / f"{slug}_{tag}_{yr}.png"

        print(f"  ↳ Export GeoTIFF : {tif_path.name} …", end=' ', flush=True)

        # Visualiser en RGB (0–255) avant export PNG
        img_vis = img.visualize(**VIS).clip(roi)

        try:
            geemap.ee_export_image(
                img_vis,
                filename=str(tif_path),
                scale=scale,
                region=roi,
                file_per_band=False
            )
            print("✅")
        except Exception as e:
            print(f"⚠️  Erreur : {e}")
            continue

        # Conversion GeoTIFF → PNG via GDAL (si disponible) ou rasterio
        print(f"  ↳ Conversion PNG  : {png_path.name} …", end=' ', flush=True)
        try:
            import rasterio
            from rasterio.plot import reshape_as_image
            import numpy as np
            from PIL import Image as PILImage

            with rasterio.open(tif_path) as src:
                data = src.read()          # (bands, rows, cols)
                rgb  = reshape_as_image(data[:3])  # (rows, cols, 3)
                rgb  = np.clip(rgb, 0, 255).astype('uint8')

            img_pil = PILImage.fromarray(rgb)
            # Cible : 300 dpi sur A4 paysage (297 × 210 mm)
            # → 3508 × 2480 px ; on redimensionne en conservant le ratio
            img_pil.thumbnail((3508, 2480), PILImage.LANCZOS)
            img_pil.save(str(png_path), dpi=(300, 300))
            print("✅")
        except ImportError:
            print("⚠️  rasterio/Pillow non installé – PNG ignoré")
        except Exception as e:
            print(f"⚠️  Erreur PNG : {e}")


def export_to_drive(meta, folder='LandsatExports', scale=30):
    """
    Alternative : export asynchrone vers Google Drive (pour très grandes zones).
    Surveiller la progression dans https://code.earthengine.google.com/tasks
    """
    slug = safe_filename(meta['site_name'])
    roi  = meta['roi']

    for tag, img, yr in [
        ('avant', meta['img_b'], meta['yr_b']),
        ('apres', meta['img_a'], meta['yr_a']),
    ]:
        task = ee.batch.Export.image.toDrive(
            image       = img.visualize(**VIS).clip(roi),
            description = f"{slug}_{tag}_{yr}",
            folder      = folder,
            fileNamePrefix = f"{slug}_{tag}_{yr}",
            region      = roi,
            scale       = scale,
            crs         = 'EPSG:4326',
            maxPixels   = 1e9
        )
        task.start()
        print(f"  ↳ Tâche Drive lancée : {slug}_{tag}_{yr} (id={task.id})")


print(f"Répertoire d'export : {EXPORT_DIR.resolve()}")

In [ ]:
# ── Export de tous les sites ───────────────────────────────────────────────────
# Par défaut : export local (GeoTIFF + PNG 300 dpi).
# Remplacer export_site_images par export_to_drive pour passer par Google Drive.

all_metas = [
    meta_carrare,
    meta_saleve,
    meta_arana,
    meta_msh,
    meta_siam,
    meta_maldives,
    meta_geneve,
    meta_singapour,
    meta_lagos,
    meta_catba,
]

for meta in all_metas:
    print(f"\n📦  {meta['site_name']}")
    export_site_images(meta, scale=30)

print("\n✅  Export terminé. Fichiers dans :", EXPORT_DIR.resolve())

---
## 7. 🎞️ GIF Timelapse (1984 → présent)

Pour chaque site, un GIF animé est généré en utilisant `geemap.landsat_timelapse()`, qui s'appuie sur les collections Landsat annuelles disponibles dans GEE.

Les GIF sont sauvegardés dans `./exports/timelapse/`.

**Paramètres clés :**
- `bands` : RGB naturel adapté à Landsat 5/7/8/9
- `dimensions` : largeur en pixels (hauteur calculée automatiquement)
- `frames_per_second` : vitesse d'animation
- `title` : texte affiché sur le GIF (nom du site)
- `apply_fmask` : masque les nuages et les ombres

In [ ]:
from IPython.display import Image as IPyImage, display

TIMELAPSE_DIR = EXPORT_DIR / 'timelapse'
TIMELAPSE_DIR.mkdir(exist_ok=True)


def make_timelapse(meta, fps=3, width_px=600):
    """
    Génère un GIF timelapse Landsat pour un site donné.

    Paramètres
    ----------
    meta      : dict retourné par make_split_map()
    fps       : images par seconde dans le GIF
    width_px  : largeur du GIF en pixels (hauteur proportionnelle)
    """
    slug     = safe_filename(meta['site_name'])
    gif_path = str(TIMELAPSE_DIR / f"{slug}_timelapse.gif")
    roi      = meta['roi']

    # Calculer les années de début/fin
    start_year = meta['yr_b']
    end_year   = meta['yr_a']

    # Ratio largeur/hauteur à partir de l'emprise
    ratio  = meta['half_w_m'] / meta['half_h_m']
    height_px = int(width_px / ratio)

    print(f"  🎞️  Timelapse {start_year}–{end_year} ({width_px}×{height_px} px) …",
          end=' ', flush=True)

    try:
        geemap.landsat_timelapse(
            roi             = roi,
            out_gif         = gif_path,
            start_year      = start_year,
            end_year        = end_year,
            start_date      = '04-01',
            end_date        = '10-31',
            bands           = ['Red', 'Green', 'Blue'],   # RGB naturel
            vis_params      = VIS,
            dimensions      = width_px,
            frames_per_second = fps,
            title           = meta['site_name'],
            apply_fmask     = True,        # masque nuages + ombres
            satellite_color = 'white',
            add_progress_bar= True,
        )
        print("✅")
        return gif_path
    except Exception as e:
        print(f"⚠️  Erreur : {e}")
        return None


print(f"Répertoire timelapse : {TIMELAPSE_DIR.resolve()}")

In [ ]:
# ── Génération de tous les timelapses ─────────────────────────────────────────
# Note : chaque appel peut prendre 1–3 minutes selon la taille de la zone.

gif_paths = {}

for meta in all_metas:
    print(f"\n🎬  {meta['site_name']}")
    gif = make_timelapse(meta, fps=3, width_px=600)
    if gif:
        gif_paths[meta['site_name']] = gif

print(f"\n✅  {len(gif_paths)}/{len(all_metas)} timelapse(s) générés.")

In [ ]:
# ── Affichage inline des GIF dans le notebook ──────────────────────────────────

for site_name, gif_path in gif_paths.items():
    print(f"\n🎞️  {site_name}")
    display(IPyImage(filename=gif_path, width=600))

---
## 8. Notes méthodologiques

| Paramètre | Valeur |
|-----------|--------|
| **Satellites** | Landsat 5 TM (1984–1998), Landsat 7 ETM+ (1999–2012), Landsat 8/9 OLI (2013–) |
| **Collection** | USGS Landsat Collection 2, Tier 1, Level-2 Surface Reflectance |
| **Résolution** | 30 m (natif Landsat) |
| **Fenêtre temporelle** | Avril – Octobre (minimiser les nuages et la neige) |
| **Filtrage nuages** | CLOUD_COVER < 20 % (fallback 40 % sur l'année complète) |
| **Composite** | Médiane de la collection filtrée |
| **Visualisation** | Couleurs naturelles (RGB) ; min=0.0, max=0.3, gamma=1.4 |
| **Année "avant"** | Première année avec images disponibles à partir de l'année indiquée (Landsat 5/7 ; avance jusqu'à 5 ans si nuageux) |
| **Année "après"** | Année la plus récente disponible dans GEE (Landsat 8/9 ; remontée jusqu'à 5 ans si nuageux) |
| **ROI** | Rectangle centré sur le site ; dimensions adaptées à chaque site |
| **Export** | GeoTIFF 30 m + PNG 300 dpi (A4 paysage) via `rasterio` + `Pillow` |
| **Timelapse** | `geemap.landsat_timelapse()` ; GIF animé 600 px, 3 fps, masque nuages |

### Limites
- La résolution Landsat (30 m) est à la limite de détection pour les petits aménagements (Baby-Plage de Genève).
- Certains sites (Mont-Saint-Hilaire) évoluent principalement en profondeur : l'empreinte surfacique peut sembler stable.
- Les timelapse sont sensibles à la disponibilité des images (certaines années peuvent être sautées si nuageuses).

### Pour aller plus loin
- Utiliser **Sentinel-2** (10 m) pour les sites à faible emprise surfacique.
- Calculer des indices spectraux (NDVI, NDWI, MNDWI) pour quantifier les changements de végétation ou de surfaces en eau.
- Exporter en **COG** (Cloud Optimized GeoTIFF) pour partage et publication web.